# Polyp detection across hospitals
Train YOLO11 on Kvasir-SEG (Norway), test on CVC-ClinicDB (Barcelona), and choose a screening threshold.

Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!nvidia-smi -L

## 1. Get the code

In [ ]:
!git clone -q https://github.com/mina-li2/polyp-detection-domain-shift.git
%cd /content/polyp-detection-domain-shift
!pip install -q -r requirements.txt

## 2. Get the data
Download `kvasir-seg.zip` from https://datasets.simula.no/kvasir-seg/ and upload it when the cell asks. CVC-ClinicDB is downloaded from a Hugging Face mirror (official page: https://polyp.grand-challenge.org/CVCClinicDB/).

In [ ]:
!mkdir -p raw
from google.colab import files
files.upload()   # select kvasir-seg.zip
!mv kvasir-seg.zip raw/ && unzip -oq raw/kvasir-seg.zip -d raw/

from huggingface_hub import snapshot_download
snapshot_download(repo_id='berkaytrhn/cvc-clinicdb', repo_type='dataset',
                  local_dir='raw/cvc', allow_patterns=['CVC-ClinicDB/**'])
!ls raw/Kvasir-SEG raw/cvc/CVC-ClinicDB

## 3. Convert masks to YOLO boxes

In [ ]:
!python prepare_data.py --kvasir raw/Kvasir-SEG --cvc raw/cvc/CVC-ClinicDB

## 4. Train
About 8–13 minutes each on a T4.

In [ ]:
!python train.py --exp baseline --epochs 50

In [ ]:
!python train.py --exp strongaug --epochs 50

In [ ]:
!python train.py --exp finetune

In [ ]:
# Optional: back up to Google Drive in case the session disconnects
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/polyp-detection && cp -r runs /content/drive/MyDrive/polyp-detection/

## 5. Same hospital vs new hospital

In [ ]:
!python evaluate.py
from IPython.display import Image, display
display(Image('results/domain_shift.png'))
display(Image('results/cvc_examples_baseline.jpg'))

## 6. Screening threshold: recall vs false alarms

In [ ]:
!python threshold_sweep.py
import pandas as pd
d = pd.read_csv('results/threshold_sweep.csv')
print(d[d.threshold <= 0.25].to_string(index=False))
display(Image('results/recall_vs_false_alarms.png'))

## 7. Demo video (threshold 0.10)

In [ ]:
!python demo.py --weights runs/baseline/weights/best.pt --source data/yolo/cvc/images/test --conf 0.10

In [ ]:
!zip -rq results.zip results
files.download('results.zip')